# Escherichia Coli exploration

**Author: T. Ruokokoski**

This notebook loads and inspects *E. Coli iML1515*-model.

Organism: Escherichia coli str. K-12 substr. MG1655

Model: [iML1515](http://bigg.ucsd.edu/models/iML1515)

- Display interactive map of reactions (only core model available for Escher)
- Run FBA to determine default maximum biomass production.
- Compare FBA and pFBA
- Display all exhange (EX) reactions
- Display all demand (DM) reactions
- Simulate anaerobic conditions by disabling oxygen uptake.
- Identify essential carbon sources by disabling them one at a time.
- Test alternative nitrogen sources
- Identify which base exchanges are essential for growth.
- Test which individual carbon sources (e.g. glucose, lactate, acetate) support growth.
- Inspect details of all reactions
- Inspect specific reactions and metabolites
- Display all reactions
- Run FVA on exchange reactions

In [1]:
import os
from cobra.io import load_model, read_sbml_model
from cobra import Model
from cobra.flux_analysis import pfba, flux_variability_analysis
from cobra.medium import minimal_medium
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings("ignore", message="Solver status is 'infeasible'")

# Set paths
model_dir = "./models"

pd.set_option('display.max_rows', None)

### Visualizing Core Metabolism

This section loads the *E. coli core* metabolic model and visualizes it using an interactive Escher map. The map shows pathways and allows inspection of reaction fluxes.

In [2]:
from escher import Builder
from cobra.io import to_json
import json

# Read model from file
model = read_sbml_model(os.path.join(model_dir, "iML1515.xml"))

# Create Escher builder
b = Builder(
    map_name='e_coli_core.Core metabolism',  # only available Escher map
    model_json=to_json(model)                # use iML1515 reactions
)
b


Builder()

### Basic Model Summary

Print statistics: number of reactions, metabolites, and genes for each model.

In [3]:
print(f"Model: {model.id}")
print(f"Reactions: {len(model.reactions)}") # biochemical transformations
print(f"Metabolites: {len(model.metabolites)}") # chemical compounds involved
print(f"Genes: {len(model.genes)}") # genes associated with enzymes that catalyze reactions
display(model.summary())

Model: iML1515
Reactions: 2712
Metabolites: 1877
Genes: 1516


Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.004565,0,0.00%
cl_e,EX_cl_e,0.004565,0,0.00%
cobalt2_e,EX_cobalt2_e,2.192E-05,0,0.00%
cu2_e,EX_cu2_e,0.0006218,0,0.00%
fe2_e,EX_fe2_e,0.01409,0,0.00%
glc__D_e,EX_glc__D_e,10,6,100.00%
k_e,EX_k_e,0.1712,0,0.00%
mg2_e,EX_mg2_e,0.007608,0,0.00%
mn2_e,EX_mn2_e,0.000606,0,0.00%
mobd_e,EX_mobd_e,6.139E-06,0,0.00%


### Run Flux Balance Analysis (FBA)

Perform FBA to compute the maximum biomass production rate under default model constraints. This simulates optimal growth conditions and reveals the most active reactions in the metabolic network.

In [4]:
# Run Flux Balance Analysis (FBA)
default_solution = model.optimize()

# Default objective function is biomass production reaction
print(f"\nMax biomass growth rate: {default_solution.objective_value:.4f} mmol/gDW/hour")

active_dm = [r.id for r in model.reactions
                if r.id.startswith(('DM_')) and abs(default_solution.fluxes[r.id]) > 1e-8]
print('Active DM:', active_dm)

# Get top 10 flux-carrying reactions
top_fluxes = default_solution.fluxes.sort_values(ascending=False).head(10)

top_reactions = []

for rxn_id, flux in top_fluxes.items():
    rxn = model.reactions.get_by_id(rxn_id)
    top_reactions.append({
        "Reaction ID": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.reaction,
        "Flux": round(flux, 4)
    })

df_top = pd.DataFrame(top_reactions)
print("\nTop flux-carrying reactions:")
display(df_top)



Max biomass growth rate: 0.8770 mmol/gDW/hour
Active DM: ['DM_amob_c', 'DM_5drib_c', 'DM_4crsol_c']

Top flux-carrying reactions:


,Reaction ID,Name,Equation,Flux
0,ATPS4rpp,ATP synthase (four protons for one ATP) (perip...,adp_c + 4.0 h_p + pi_c <=> atp_c + h2o_c + 3.0...,70.4325
1,EX_h2o_e,H2O exchange,h2o_e <=>,47.1624
2,CYTBO3_4pp,Cytochrome oxidase bo3 (ubiquinol-8: 4 protons...,4.0 h_c + 0.5 o2_c + q8h2_c --> h2o_c + 4.0 h_...,44.2563
3,NADH16pp,NADH dehydrogenase (ubiquinone-8 & 3 protons) ...,4.0 h_c + nadh_c + q8_c --> 3.0 h_p + nad_c + ...,37.9970
4,EX_co2_e,CO2 exchange,co2_e <=>,24.0033
5,O2tex,Oxygen transport via diffusion (extracellular ...,o2_e <=> o2_p,22.1318
6,O2tpp,O2 transport via diffusion (periplasm),o2_p <=> o2_c,22.1318
7,GAPD,Glyceraldehyde-3-phosphate dehydrogenase,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c,17.1054
8,ENO,Enolase,2pg_c <=> h2o_c + pep_c,15.5989
9,GLCtex_copy1,Glucose transport via diffusion (extracellular...,glc__D_e --> glc__D_p,10.0000


### Parsimonius Flux Balance Analysis (pFBA)

pFBA finds a flux distribution which gives the optimal growth rate, but minimizes the total sum of flux. Both pFBA and FBA should return identical results within solver tolerances for the objective being optimized.

In [5]:
pfba_solution = pfba(model)

diff = abs(default_solution.fluxes["BIOMASS_Ec_iML1515_core_75p37M"] - pfba_solution.fluxes["BIOMASS_Ec_iML1515_core_75p37M"])
print(f"Difference in between standard FBA and pFBA: {diff}")

Difference in between standard FBA and pFBA: 1.4210854715202004e-14


### Exchange Reactions Overview

List and inspect all exchange reactions, which represent metabolite uptake and secretion between the cell and its environment. Useful for understanding how the model interfaces with external nutrients and byproducts.

In [6]:
print(f"Number of exchange reactions: {len(model.exchanges)}")
exchange_data = []

for rxn in model.reactions:
    if rxn.id.startswith('EX'):
        exchange_data.append({
            "Reaction ID": rxn.id,
            "Name": rxn.name,
            "Bounds": rxn.bounds,
            "Equation": rxn.reaction,
            "Flux": round(default_solution.fluxes[rxn.id], 4)
        })

df = pd.DataFrame(exchange_data)
display(df)

Number of exchange reactions: 331


,Reaction ID,Name,Bounds,Equation,Flux
0,EX_pi_e,Phosphate exchange,"(-1000.0, 1000.0)",pi_e <=>,-0.8460
1,EX_co2_e,CO2 exchange,"(-1000.0, 1000.0)",co2_e <=>,24.0033
2,EX_met__L_e,L-Methionine exchange,"(0.0, 1000.0)",met__L_e -->,0.0000
3,EX_metsox_S__L_e,L-Methionine S-oxide exchange,"(0.0, 1000.0)",metsox_S__L_e -->,0.0000
4,EX_acgam_e,N-Acetyl-D-glucosamine exchange,"(0.0, 1000.0)",acgam_e -->,0.0000
5,EX_cellb_e,Cellobiose exchange,"(0.0, 1000.0)",cellb_e -->,0.0000
6,EX_crn_e,L-Carnitine exchange,"(0.0, 1000.0)",crn_e -->,0.0000
7,EX_hxan_e,Hypoxanthine exchange,"(0.0, 1000.0)",hxan_e -->,0.0000
8,EX_ile__L_e,L-Isoleucine exchange,"(0.0, 1000.0)",ile__L_e -->,0.0000
9,EX_chol_e,Choline exchange,"(0.0, 1000.0)",chol_e -->,0.0000


In [7]:
exchange_ids = [rxn.id for rxn in model.exchanges]
print(exchange_ids)

['EX_pi_e', 'EX_co2_e', 'EX_met__L_e', 'EX_metsox_S__L_e', 'EX_acgam_e', 'EX_cellb_e', 'EX_crn_e', 'EX_hxan_e', 'EX_ile__L_e', 'EX_chol_e', 'EX_fe3_e', 'EX_lac__L_e', 'EX_leu__L_e', 'EX_glcn_e', 'EX_no3_e', 'EX_h_e', 'EX_orn_e', 'EX_gln__L_e', 'EX_pro__L_e', 'EX_glyc_e', 'EX_man_e', 'EX_ade_e', 'EX_mn2_e', 'EX_4abut_e', 'EX_ac_e', 'EX_akg_e', 'EX_ala__L_e', 'EX_arg__L_e', 'EX_asp__L_e', 'EX_pyr_e', 'EX_succ_e', 'EX_thymd_e', 'EX_rib__D_e', 'EX_tyr__L_e', 'EX_cytd_e', 'EX_dcyt_e', 'EX_fum_e', 'EX_sbt__D_e', 'EX_glu__L_e', 'EX_gua_e', 'EX_btn_e', 'EX_ptrc_e', 'EX_spmd_e', 'EX_thym_e', 'EX_xtsn_e', 'EX_fe2_e', 'EX_glc__D_e', 'EX_alltn_e', 'EX_ura_e', 'EX_val__L_e', 'EX_xan_e', 'EX_dgsn_e', 'EX_arab__L_e', 'EX_fru_e', 'EX_gal_e', 'EX_xyl__D_e', 'EX_duri_e', 'EX_for_e', 'EX_gly_e', 'EX_h2_e', 'EX_lys__L_e', 'EX_ser__L_e', 'EX_thm_e', 'EX_trp__L_e', 'EX_din_e', 'EX_fmn_e', 'EX_gthox_e', 'EX_tmao_e', 'EX_acald_e', 'EX_melib_e', 'EX_sucr_e', 'EX_tre_e', 'EX_zn2_e', 'EX_hdcea_e', 'EX_lac__D_e',

### Demand (DM) Reactions Overview

List and inspect all demand reactions.

In [8]:
# collect all DM reactions
dm_reactions = [rxn for rxn in model.reactions if rxn.id.startswith(("DM_"))]
dm_data =[]

for rxn in dm_reactions:
    dm_data.append({
        "Reaction ID": rxn.id,
        "Name": rxn.name,
        "Bounds": rxn.bounds,
        "Equation": rxn.reaction,
        "Flux": round(default_solution.fluxes[rxn.id], 4)
    })

df = pd.DataFrame(dm_data)
display(df)

,Reaction ID,Name,Bounds,Equation,Flux
0,DM_amob_c,Sink needed to allow S-Adenosyl-4-methylthio-2...,"(0.0, 1000.0)",amob_c -->,0.0000
1,DM_5drib_c,Sink needed to allow 5'-deoxyribose to leave s...,"(0.0, 1000.0)",5drib_c -->,0.0002
2,DM_oxam_c,Sink needed to allow oxamate to leave system,"(0.0, 1000.0)",oxam_c -->,0.0000
3,DM_aacald_c,Sink needed to allow aminoacetaldehyde to leav...,"(0.0, 1000.0)",aacald_c -->,0.0000
4,DM_4crsol_c,Sink needed to allow p-Cresol to leave system,"(0.0, 1000.0)",4crsol_c -->,0.0002
5,DM_mththf_c,"Sink needed to allow (2R,4S)-2-methyl-2,3,3,4-...","(0.0, 1000.0)",mththf_c -->,0.0000


### Default medium

In [9]:
medium_rxns = list(model.medium.keys())
print(medium_rxns)

model.medium

['EX_pi_e', 'EX_co2_e', 'EX_fe3_e', 'EX_h_e', 'EX_mn2_e', 'EX_fe2_e', 'EX_glc__D_e', 'EX_zn2_e', 'EX_mg2_e', 'EX_ca2_e', 'EX_ni2_e', 'EX_cu2_e', 'EX_sel_e', 'EX_cobalt2_e', 'EX_h2o_e', 'EX_mobd_e', 'EX_so4_e', 'EX_nh4_e', 'EX_k_e', 'EX_na1_e', 'EX_cl_e', 'EX_o2_e', 'EX_tungs_e', 'EX_slnt_e']


{'EX_pi_e': 1000.0,
 'EX_co2_e': 1000.0,
 'EX_fe3_e': 1000.0,
 'EX_h_e': 1000.0,
 'EX_mn2_e': 1000.0,
 'EX_fe2_e': 1000.0,
 'EX_glc__D_e': 10.0,
 'EX_zn2_e': 1000.0,
 'EX_mg2_e': 1000.0,
 'EX_ca2_e': 1000.0,
 'EX_ni2_e': 1000.0,
 'EX_cu2_e': 1000.0,
 'EX_sel_e': 1000.0,
 'EX_cobalt2_e': 1000.0,
 'EX_h2o_e': 1000.0,
 'EX_mobd_e': 1000.0,
 'EX_so4_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_k_e': 1000.0,
 'EX_na1_e': 1000.0,
 'EX_cl_e': 1000.0,
 'EX_o2_e': 1000.0,
 'EX_tungs_e': 1000.0,
 'EX_slnt_e': 1000.0}

### Find minimal medium

In [10]:

model.objective = 'BIOMASS_Ec_iML1515_core_75p37M'

solution = model.optimize()
print("Growth rate:", solution.objective_value)

min_med = minimal_medium(model, solution.objective_value)
print(min_med)
len(min_med)


Growth rate: 0.876997214426972
EX_pi_e          0.845957
EX_mn2_e         0.000606
EX_fe2_e         0.014085
EX_glc__D_e     10.000000
EX_zn2_e         0.000299
EX_mg2_e         0.007608
EX_ca2_e         0.004565
EX_ni2_e         0.000283
EX_cu2_e         0.000622
EX_cobalt2_e     0.000022
EX_mobd_e        0.000006
EX_so4_e         0.220845
EX_nh4_e         9.471495
EX_k_e           0.171184
EX_cl_e          0.004565
EX_o2_e         22.131763
dtype: float64


16

### Set realistic base

In [11]:
base_medium = {
    # Major nutrients
    'EX_nh4_e': 1000.0,    # Nitrogen
    'EX_pi_e': 1000.0,     # Phosphate
    'EX_so4_e': 1000.0,    # Sulfate

    # Cations / metals
    'EX_mg2_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_ca2_e': 1000.0,
    'EX_na1_e': 1000.0,
    'EX_ni2_e': 1000.0,
    'EX_cl_e': 1000.0,
    'EX_fe2_e': 1000.0,
    'EX_zn2_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    'EX_co2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_mobd_e': 1000.0,

    # Water and protons
    'EX_h2o_e': 1000.0,
    'EX_h_e': 1000.0,

    # Oxygen for aerobic growth
    'EX_o2_e': 1000.0
}

model.medium = base_medium
len(base_medium)

19

### Find carbon sources

In [12]:
aa_candidates = ['EX_ala__L_e', 'EX_pro__L_e', 'EX_thr__L_e', 'EX_gly_e'] # amino acids in Faure experiments
good_aa_sources = []

carbon_exchanges = []
for rxn in model.exchanges:
    if rxn.id in base_medium:
        continue
    for met in rxn.metabolites:
        if met.formula and 'C' in met.formula:
            carbon_exchanges.append(rxn.id)
            break

growth_threshold = 0.05
carbon_sources = []
results = []
for rxn_id in carbon_exchanges:
    for r in carbon_exchanges:
        model.reactions.get_by_id(r).bounds = (0.0, 1000.0)
    
    model.medium = base_medium.copy()
    
    model.reactions.get_by_id(rxn_id).bounds = (-10.0, 1000.0)
    
    sol = model.optimize()
    if sol.objective_value > growth_threshold:
        carbon_sources.append((rxn_id, sol.objective_value))
        results.append({
            "Carbon source": rxn_id,
            "Biomass Flux": round(sol.fluxes['BIOMASS_Ec_iML1515_core_75p37M'], 4)
        })
        if rxn_id in aa_candidates:
            good_aa_sources.append(rxn_id)

    model.reactions.get_by_id(rxn_id).bounds = (0, 1000.0)

print(f"Carbon sources found: {len(carbon_sources)}\n")
print("Amino acids that worked as carbon sources:", good_aa_sources)

df_carbons = pd.DataFrame(results)
display(df_carbons)

Carbon sources found: 185

Amino acids that worked as carbon sources: ['EX_pro__L_e', 'EX_ala__L_e', 'EX_gly_e', 'EX_thr__L_e']


,Carbon source,Biomass Flux
0,EX_acgam_e,1.1313
1,EX_cellb_e,1.7798
2,EX_hxan_e,0.2887
3,EX_chol_e,0.0614
4,EX_lac__L_e,0.3506
5,EX_glcn_e,0.7971
6,EX_orn_e,0.7029
7,EX_gln__L_e,0.6164
8,EX_pro__L_e,0.7138
9,EX_glyc_e,0.4947


In [13]:
# test one carbon source to verify
model.reactions.get_by_id('EX_LalaLglu_e').bounds = (-10.0, 1000.0)

sol = model.optimize()
print(f"Growth on L-Ala-L-Glu only: {sol.objective_value:.4f}")

model.reactions.get_by_id('EX_LalaLglu_e').bounds = (0, 1000.0)
model.reactions.get_by_id('EX_glc__D_e').bounds = (-10.0, 1000.0)


Growth on L-Ala-L-Glu only: 1.0436


### Anaerobic Growth Test

Simulate growth under anaerobic conditions by disabling oxygen uptake and re-running FBA. This tests whether the model can produce biomass without oxygen and reveals how flux distribution changes in its absence.

In [14]:
# Disable oxygen uptake
oxygen = model.reactions.get_by_id('EX_o2_e')
oxygen_lb_backup = oxygen.lower_bound
oxygen.lower_bound = 0.0

# Re-run FBA with no oxygen
anaerobic_solution = model.optimize()

print("\n=== Anaerobic growth test (oxygen uptake disabled) ===")
if anaerobic_solution.status == 'optimal':
    print(f"Biomass without oxygen: {anaerobic_solution.objective_value:.4f}")
    
    # Get top 10 flux-carrying reactions under anaerobic conditions
    top_fluxes_anaerobic = anaerobic_solution.fluxes.sort_values(ascending=False).head(10)

    anaerobic_reactions = []

    for rxn_id, flux in top_fluxes_anaerobic.items():
        rxn = model.reactions.get_by_id(rxn_id)
        anaerobic_reactions.append({
            "Reaction ID": rxn.id,
            "Name": rxn.name,
            "Equation": rxn.reaction,
            "Flux": round(flux, 4)
        })

    df_anaerobic = pd.DataFrame(anaerobic_reactions)
    print("\nTop flux-carrying reactions under anaerobic conditions:")
    display(df_anaerobic)

else:
    print("Optimization failed: no feasible solution without oxygen.")

# Restore original oxygen bound
oxygen.lower_bound = oxygen_lb_backup


=== Anaerobic growth test (oxygen uptake disabled) ===
Biomass without oxygen: 0.0000

Top flux-carrying reactions under anaerobic conditions:


,Reaction ID,Name,Equation,Flux
0,GAPD,Glyceraldehyde-3-phosphate dehydrogenase,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c,6.86
1,ENO,Enolase,2pg_c <=> h2o_c + pep_c,6.86
2,EX_etoh_e,Ethanol exchange,etoh_e -->,6.86
3,EX_co2_e,CO2 exchange,co2_e <=>,6.86
4,PDH,Pyruvate dehydrogenase,coa_c + nad_c + pyr_c --> accoa_c + co2_c + na...,6.86
5,ATPM,ATP maintenance requirement,atp_c + h2o_c --> adp_c + h_c + pi_c,6.86
6,PGI,Glucose-6-phosphate isomerase,g6p_c <=> f6p_c,3.43
7,GLCptspp,D-glucose transport via PEP:Pyr PTS (periplasm),glc__D_p + pep_c --> g6p_c + pyr_c,3.43
8,GLCtex_copy1,Glucose transport via diffusion (extracellular...,glc__D_e --> glc__D_p,3.43
9,TPI,Triose-phosphate isomerase,dhap_c <=> g3p_c,3.43


### Test Growth with Alternative Nitrogen Sources

This test evaluates whether E. coli can grow using different nitrogen sources (ammonia, glutamine, or glutamate) while keeping a fixed carbon source (glucose) and essential base exchanges enabled. Each nitrogen source is tested individually by setting its uptake and disabling all others.

In [15]:
# TODO: modify this for iML1515


# Save current medium and model state
original_medium = model.medium.copy()

carbon_source = 'EX_glc__D_e'

nitrogen_sources = [
    'EX_gln__L_e',   # L-Glutamine
    'EX_glu__L_e',   # L-Glutamate
    'EX_nh4_e'       # Ammonia
]
base_exchanges = [
    'EX_co2_e',
    'EX_h_e',
    'EX_h2o_e',
    'EX_o2_e',
    'EX_pi_e'
]

print("Testing nitrogen source alternatives with fixed carbon source:", carbon_source)
survival_results = {}

for nitrogen in nitrogen_sources:
    for rxn in model.exchanges:
        rxn.lower_bound = 0.0

    # Enable one carbon source
    model.reactions.get_by_id(carbon_source).lower_bound = -10

    # Enable base exchanges
    for ex in base_exchanges:
        model.reactions.get_by_id(ex).lower_bound = -1000

    # Enable only current nitrogen source
    model.reactions.get_by_id(nitrogen).lower_bound = -10

    solution = model.optimize()
    survived = solution.status == 'optimal'
    survival_results[nitrogen] = solution.objective_value if survived else 0.0

print("\nSurvival (growth) with each nitrogen source:")
for nit, growth in survival_results.items():
    status = f"YES (growth = {growth:.4f})" if growth > 0 else "NO"
    print(f"  {nit}: {status}")

# Restore original medium
model.medium = original_medium.copy()

Testing nitrogen source alternatives with fixed carbon source: EX_glc__D_e

Survival (growth) with each nitrogen source:
  EX_gln__L_e: NO
  EX_glu__L_e: YES (growth = 0.0000)
  EX_nh4_e: YES (growth = 0.0000)


### Growth sensitivity to nutrient uptake

Tests how varying uptake of each non-carbon nutrient affects growth.  
Stores growth rates in `growth_profiles` and restores original bounds after testing.


In [16]:
uptake_exchanges = [
    rxn for rxn in model.exchanges
    if rxn.lower_bound < 0 and rxn.id not in carbon_exchanges
]

uptake_levels = np.linspace(0, 10, 5)
growth_profiles = {}

for rxn in uptake_exchanges:
    rxn_id = rxn.id
    original_lb = rxn.lower_bound
    growths = []

    for level in uptake_levels:
        rxn.lower_bound = -level
        sol = model.optimize()
        growth = sol.objective_value if sol.status == 'optimal' and sol.objective_value else 0.0
        growths.append(growth)

    growth_profiles[f"{rxn_id} ({rxn.name})"] = growths
    rxn.lower_bound = original_lb

df_growth = pd.DataFrame(growth_profiles, index=[f"{lvl:.1f}" for lvl in uptake_levels])
df_growth.index.name = "Uptake (mmol/gDW/h)"
display(df_growth.T.round(4))

Uptake (mmol/gDW/h),0.0,2.5,5.0,7.5,10.0
EX_pi_e (Phosphate exchange),-0.000,0.8770,0.8770,0.8770,0.8770
EX_co2_e (CO2 exchange),0.877,0.8770,0.8770,0.8770,0.8770
EX_h_e (H+ exchange),0.877,0.8770,0.8770,0.8770,0.8770
EX_mn2_e (Mn2+ exchange),0.000,0.8770,0.8770,0.8770,0.8770
EX_fe2_e (Fe2+ exchange),-0.000,0.8770,0.8770,0.8770,0.8770
EX_zn2_e (Zinc exchange),-0.000,0.8770,0.8770,0.8770,0.8770
EX_mg2_e (Mg exchange),-0.000,0.8770,0.8770,0.8770,0.8770
EX_ca2_e (Calcium exchange),-0.000,0.8770,0.8770,0.8770,0.8770
EX_ni2_e (Ni2+ exchange),-0.000,0.8770,0.8770,0.8770,0.8770
EX_cu2_e (Cu2+ exchange),-0.000,0.8770,0.8770,0.8770,0.8770


### Inspect a Specific Reaction

Access a single reaction from the model by its index or ID.

In [23]:
model.reactions[2669] # inspect by index number

#BIOMASS_Ec_iML1515_core_75p37M

#model.reactions.get_by_id("EX_thr__L_e")

Reaction identifier,BIOMASS_Ec_iML1515_WT_75p37M
Name,E. coli biomass objective function (iML1515) - WT - with 75.37 GAM estimate
Memory address,0x24a3aa99810
Stoichiometry,0.000223 10fthf_c + 0.000223 2dmmql8_c + 2.5e-05 2fe2s_c + 0.000248 4fe4s_c + 0.000223 5mthf_c + 0.000279 accoa_c + 0.000223 adocbl_c + 0.499149 ala__L_c + 0.000223 amet_c + 0.28742 arg__L_c +... 0.000223 10-Formyltetrahydrofolate + 0.000223 2-Demethylmenaquinol 8 + 2.5e-05 [2Fe-2S] iron-sulfur cluster + 0.000248 [4Fe-4S] iron-sulfur cluster + 0.000223 5-Methyltetrahydrofolate + 0.000279...
GPR,
Lower bound,0.0
Upper bound,1000.0


### Inspect a Specific Metabolite

Retrieve a metabolite from the model using its ID (`"o2_e"` for extracellular oxygen) or index.

In [18]:
model.metabolites.get_by_id("o2_e")

#model.metabolites[3] # inspect by index number

Metabolite identifier,o2_e
Name,O2 O2
Memory address,0x24a38c7d8d0
Formula,O2
Compartment,e
In 2 reaction(s),"EX_o2_e, O2tex"


### Full Reaction Overview

All reactions in the model are collected into a DataFrame with basic metadata: reaction ID, name, equation, flux bounds, and associated genes. This provides a structured overview for further inspection or export.


In [19]:
reaction_data = []
reaction_ids = []

for rxn in model.reactions:
    reaction_ids.append(rxn.id)
    reaction_data.append({
        "Reaction ID": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.reaction,
        "Bounds (mmol/gDW/h)": rxn.bounds,
        #"Subsystem": getattr(rxn, "subsystem", "N/A"),
        "Gene Associations": ", ".join(g.id for g in rxn.genes) if rxn.genes else "None"
    })

df_reactions = pd.DataFrame(reaction_data)
display(df_reactions)

# print all reactions as a list
print(reaction_ids)

,Reaction ID,Name,Equation,Bounds (mmol/gDW/h),Gene Associations
0,CYTDK2,Cytidine kinase (GTP),cytd_c + gtp_c --> cmp_c + gdp_c + h_c,"(0.0, 1000.0)",b2066
1,XPPT,Xanthine phosphoribosyltransferase,prpp_c + xan_c --> ppi_c + xmp_c,"(0.0, 1000.0)",b0238
2,HXPRT,Hypoxanthine phosphoribosyltransferase (Hypoxa...,hxan_c + prpp_c --> imp_c + ppi_c,"(0.0, 1000.0)","b0125, b0238"
3,NDPK5,Nucleoside-diphosphate kinase (ATP:dGDP),atp_c + dgdp_c <=> adp_c + dgtp_c,"(-1000.0, 1000.0)","b0474, b2518"
4,SHK3Dr,Shikimate dehydrogenase,3dhsk_c + h_c + nadph_c <=> nadp_c + skm_c,"(-1000.0, 1000.0)","b3281, b1692"
5,NDPK6,Nucleoside-diphosphate kinase (ATP:dUDP),atp_c + dudp_c <=> adp_c + dutp_c,"(-1000.0, 1000.0)","b0474, b2518"
6,NDPK8,Nucleoside-diphosphate kinase (ATP:dADP),atp_c + dadp_c <=> adp_c + datp_c,"(-1000.0, 1000.0)","b0474, b2518"
7,DHORTS,Dihydroorotase,dhor__S_c + h2o_c <=> cbasp_c + h_c,"(-1000.0, 1000.0)",b1062
8,OMPDC,Orotidine-5'-phosphate decarboxylase,h_c + orot5p_c --> co2_c + ump_c,"(0.0, 1000.0)",b1281
9,PYNP2r,Pyrimidine-nucleoside phosphorylase (uracil),pi_c + uri_c <=> r1p_c + ura_c,"(-1000.0, 1000.0)",b3831


['CYTDK2', 'XPPT', 'HXPRT', 'NDPK5', 'SHK3Dr', 'NDPK6', 'NDPK8', 'DHORTS', 'OMPDC', 'PYNP2r', 'G5SD', 'CS', 'ICDHyr', 'ALATA_L2', 'DURIPP', 'ACALD', 'PTRCTA', 'ACS', 'CYSDS', 'MAN6PI', 'PPA', 'APRAUR', 'TRPAS2', 'PPCK', 'ME1', 'DB4PS', 'ALAR', 'RBFK', 'ALATA_L', 'XYLK', 'RBK', 'GLYK', 'PPM', 'ASPTA', 'ACP1_FMN', 'RBFSb', 'NDP3', 'CDPPH', 'NDP7', 'FBP', 'EX_pi_e', 'GLGC', 'ALATA_D2', 'PYK', 'SHCHD2', 'EX_co2_e', 'A5PISO', 'PMDPHT', 'CPPPGO', 'EX_met__L_e', 'GTHOr', 'ILETA', 'DHORD5', 'EX_metsox_S__L_e', 'GLYCTO2', 'VALTA', 'GLYCTO3', 'IPPMIb', 'GLYCTO4', 'ORPT', 'RBFSa', 'ACHBS', 'PFK_3', 'DHAD2', 'ACLS', 'TRPS2', 'PSCVT', 'G3PD5', 'G1PP', 'PFL', 'EX_acgam_e', 'ANS', 'FRD2', 'FRD3', 'ANPRT', 'POX', 'CHORM', 'PTAr', 'CHORS', 'IGPS', 'ACKr', 'EX_cellb_e', 'EX_crn_e', 'EX_hxan_e', 'EX_ile__L_e', 'EX_chol_e', 'EX_fe3_e', 'LEUTAi', 'EX_lac__L_e', 'EX_leu__L_e', 'EX_glcn_e', 'EX_no3_e', 'EX_h_e', 'DMATT', 'EX_orn_e', 'GRTT', 'EX_gln__L_e', 'UPP3S', 'EX_pro__L_e', 'EX_glyc_e', 'UPPDC1', 'EX_ma

### Running Flux Variablity Analysis (FVA)

Runs FVA to determine the minimum and maximum flux values that each reaction can carry while maintaining optimal growth. Useful for assessing metabolic flexibility.

In [20]:
'''

# Restore original medium
model.medium = original_medium.copy()

exchange_rxns = [rxn for rxn in model.exchanges]

# Run FVA on exchange reactions
flux_variability_analysis(model, exchange_rxns)
'''

'\n\n# Restore original medium\nmodel.medium = original_medium.copy()\n\nexchange_rxns = [rxn for rxn in model.exchanges]\n\n# Run FVA on exchange reactions\nflux_variability_analysis(model, exchange_rxns)\n'